# Genie accuracy test
Scores Genie against the answer key (12 golden questions + 5 held-out), first without instructions (baseline), then with them. Expected answers are proven in notebook 06. The eval tables are never added to the Genie space.

## Step A – Answer key

In [0]:
CREATE OR REPLACE TABLE workspace.scribd_exercise.eval_golden_questions (
  question_id STRING COMMENT 'Q1–Q12 golden, H1–H5 held-out',
  question_set STRING COMMENT 'golden or held_out',
  question STRING COMMENT 'Question asked to Genie, word for word',
  question_type STRING COMMENT 'Lookup, Breakdown, Comparison, or Trap',
  is_trap BOOLEAN COMMENT 'True if the right answer is to explain, refuse or ask',
  expected_answer STRING COMMENT 'What a correct answer contains'
)
COMMENT 'Answer key for the Genie accuracy test. Never add this table to the Genie space.';

In [0]:
INSERT INTO workspace.scribd_exercise.eval_golden_questions VALUES
('Q1',  'golden', 'What was MAU in August 2026?', 'Lookup', false, '401'),
('Q2',  'golden', 'How many active paid subscribers did each business have in August 2026?', 'Breakdown', false, 'BIZ_1 92, BIZ_2 108, BIZ_3 99; All 285 distinct people (not 299)'),
('Q3',  'golden', 'What was MRR in August 2026, and how much came from lifetime deals?', 'Lookup', false, 'USD 3,956.01; lifetime USD 959.08 (24.2%)'),
('Q4',  'golden', 'What was the churn rate for June–August 2026, by billing cycle?', 'Breakdown', false, 'All 13.2%; monthly 27.2%, annual 9.8%, lifetime 4.8%; says it is a 3-month rate'),
('Q5',  'golden', 'Which business has the highest churn?', 'Comparison', false, 'BIZ_1 15.7% (Jun–Aug 2026) but intervals overlap, so not a reliable difference; states the period'),
('Q6',  'golden', 'What share of engagement goes to new content?', 'Lookup', false, '29.2% of engagement vs. 7.7% of catalog (August 2026)'),
('Q7',  'golden', 'How good is discovery?', 'Lookup', false, 'Uses catalog reach: 75.5% (216 of 286); new 100%, recent 95.4%, older 66.3%; names the metric; no search metric'),
('Q8',  'golden', 'What was the churn rate for July 2026?', 'Trap', true, 'No monthly churn; gives May–July 2026 window 14.2% labeled as such, or asks'),
('Q9',  'golden', 'How much has churn increased?', 'Trap', true, 'Non-overlapping windows: Mar–May 10.9% vs. Jun–Aug 13.2%; intervals overlap, not a reliable increase'),
('Q10', 'golden', 'What was MRR in October 2026?', 'Trap', true, 'No number; data reliable only through August 2026'),
('Q11', 'golden', 'What''s the average session time?', 'Trap', true, 'No number; not an official metric (placeholders double the average)'),
('Q12', 'golden', 'How many users do we have?', 'Trap', true, 'Asks, or gives MAU 401 and paid subscribers 285 with definitions'),
('H1',  'held_out', 'What was churn in the first quarter of 2026?', 'Trap', true, 'Uses the Jan–Mar 2026 3-month window: 6.6% (CI 4.1–10.4%), labeled; no monthly rate or average of monthly rates'),
('H2',  'held_out', 'What''s our average revenue per paying customer?', 'Trap', true, 'Says it is not an official metric; may give MRR USD 3,956.01 and 285 paid subscribers separately. Computing about USD 13.88 without saying it is unofficial = Partial'),
('H3',  'held_out', 'How many people used the product in September 2026?', 'Trap', true, 'No number; data reliable only through August 2026'),
('H4',  'held_out', 'How many customers do we have?', 'Lookup', false, 'Active paid subscribers 285 (up to 314) as of Aug 31, 2026, naming the definition; or asks'),
('H5',  'held_out', 'Is our backlist getting seen?', 'Lookup', false, 'Catalog reach for older content: 66.3% (132 of 199) in August 2026 vs. 95–100% for newer content');

num_affected_rows,num_inserted_rows
17,17


In [0]:
SELECT question_set, COUNT(*) AS questions
FROM workspace.scribd_exercise.eval_golden_questions
GROUP BY question_set;

question_set,questions
golden,12
held_out,5


## Step B – Results
One row per Genie answer: which run (baseline or with_instructions), Genie's answer, the score and notes.

In [0]:
CREATE OR REPLACE TABLE workspace.scribd_exercise.eval_genie_results (
  run_id STRING COMMENT 'baseline (no instructions) or with_instructions',
  question_id STRING COMMENT 'Links to eval_golden_questions',
  genie_answer STRING COMMENT 'Genie''s answer, copied or summarized (key number, period, caveats)',
  score STRING COMMENT 'Correct, Partial or Wrong (rubric in golden_questions.md)',
  notes STRING COMMENT 'Why it got this score, e.g. wrong period, no caveat, invented metric'
)
COMMENT 'Genie accuracy test results. Never add this table to the Genie space.';

In [0]:
DELETE FROM workspace.scribd_exercise.eval_genie_results WHERE run_id = 'baseline';

INSERT INTO workspace.scribd_exercise.eval_genie_results VALUES
('baseline', 'Q1',  'MAU 401 in August 2026; BIZ_3 224, BIZ_1 162, BIZ_2 127; explains businesses do not sum to total', 'Correct', 'Right number, period and caveat'),
('baseline', 'Q2',  'All 285 as of Aug 31, 2026; BIZ_2 108, BIZ_3 99, BIZ_1 92; explains distinct people', 'Correct', 'Right numbers and explains why businesses do not add up'),
('baseline', 'Q3',  'MRR USD 3,956.01; lifetime USD 959.08 (24.2%); split by billing cycle and business', 'Correct', 'Right numbers and period'),
('baseline', 'Q4',  'Jun-Aug 2026 13.2% (CI 9.8-17.7%); monthly 27.2%, annual 9.8%, lifetime 4.8%, with CIs', 'Correct', 'Right numbers; labeled as the Jun-Aug period'),
('baseline', 'Q5',  'BIZ_1 highest at 15.7% for Jun-Aug 2026 (14 of 89), BIZ_2 13.0%, BIZ_3 11.1%; chart with 95% CI error bars, BIZ_1 9.6-24.7%', 'Partial', 'Right numbers, period and CIs shown, but does not say the intervals overlap so the difference may be noise'),
('baseline', 'Q6',  'New content 29.2% of engagement vs. 7.7% of catalog in August 2026; by business', 'Correct', 'Right numbers and comparison with catalog share'),
('baseline', 'Q7',  'Catalog reach 75.5% (216 of 286); new 100%, recent 95.4%, older 66.3%; says discovery is strong', 'Partial', 'Right metric and numbers, but unsupported claims: credits search and recommendations, calls older content healthy, says BIZ_3 significantly ahead without CIs'),
('baseline', 'Q8',  'Headline: Churn Rate for July 2026: 14.2% (May-Jul window, CI 10.6-18.8%); compares with August rate 13.2%', 'Partial', 'Right window and number, but calls it July churn and compares overlapping windows'),
('baseline', 'Q9',  'Churn increased dramatically from near 0% in mid-2024 to 14.2% by July 2026', 'Wrong', 'Accepts the premise, compares overlapping windows, uses tiny early bases, no CIs'),
('baseline', 'Q10', 'Not available; latest complete month is August 2026 (MRR USD 3,956.01)', 'Correct', 'No number for October'),
('baseline', 'Q11', 'No session time data in the available tables; would need event-level session data', 'Correct', 'No number; the governed views have no session column, so it cannot compute an unofficial metric'),
('baseline', 'Q12', 'MAU 401, active paid subscribers 285, active free users 109, each with a definition', 'Correct', 'Gives both definitions instead of picking one silently'),
('baseline', 'H1',  'Q1 2026 (Jan-Mar) 6.6% (CI 4.1-10.4%), 16 of 243', 'Correct', 'Uses and labels the 3-month window; no monthly rate'),
('baseline', 'H2',  'Average revenue per paying customer USD 13.88 (MRR / paid subscribers); by business', 'Partial', 'Computes an unofficial metric without saying it is not defined'),
('baseline', 'H3',  'Not available; latest month August 2026 (401 MAU)', 'Correct', 'No number for September'),
('baseline', 'H4',  '285 active paid subscribers as of Aug 31, 2026; by business', 'Partial', 'Right number and definition, but no upper bound (314)'),
('baseline', 'H5',  'Older content 66.3% reach (132 of 199) vs. 95.4% recent and 100% new; calls the backlist healthy', 'Partial', 'Right mapping and numbers, but judges a third of the backlist going unseen as healthy and calls lower engagement expected');

num_affected_rows
0


num_affected_rows,num_inserted_rows
17,17


In [0]:
SELECT score, COUNT(*) AS questions
FROM workspace.scribd_exercise.eval_genie_results
WHERE run_id = 'baseline'
GROUP BY score;

score,questions
Correct,10
Partial,6
Wrong,1


### Instructions used in run 2
SCOPE
Answer only from the metric views in this space. Every number must come from a view column; never aggregate rows into a new metric or invent one. If a metric doesn't exist, say so and suggest the closest official one.

TIME
- Data is reliable only through August 2026. For September 2026 or later, say so and give no number.
- If no period is given, use the latest complete period (August 2026; churn: the 3-month window ending August 31, 2026) and say which.
- Always state the period. Trends start in January 2023.

BUSINESSES
- For company-wide numbers, always use the business_id = 'All businesses' row; never compute totals yourself.
- MAU and active paid subscribers: businesses do NOT add up to the total (people can belong to several). MRR: businesses add up. Rates: never add or average across businesses.

CHURN
- Churn exists only as 3-month windows; there is no monthly churn. For a month, give the window ending in that month with its full label (e.g., "January–March 2026"), never "<month> churn".
- Compare only windows that don't overlap. Use billing_cycle = 'All billing cycles' unless asked.

UNCERTAINTY
- For churn and catalog reach, check the 95% intervals (_ci_low, _ci_high). If they overlap, say the difference may be chance.
- For active paid subscribers and MRR, report the upper bound too ("X, up to Y").
- Describe differences as associations. Check assumptions in the question before accepting them.

TERMS
- "Users": ask whether MAU or paid subscribers, or give both with definitions.
- "Revenue" = MRR (listed price; no collected revenue). "Discovery" = catalog reach.

NOT AVAILABLE (explain, don't calculate)
Average session time (placeholder durations double it), completion rate or funnel (completions lack opens), trial conversion (stale trials), collected revenue (no payment data), search success (no query text), monthly churn, content rankings (engagement too even).

STYLE
Number and period first, then a one-line meaning, then caveats. Percentages with one decimal; USD with two decimals.

In [0]:
INSERT INTO workspace.scribd_exercise.eval_genie_results VALUES
('with_instructions', 'Q1',  'MAU 401 in August 2026', 'Correct', 'Right number and period'),
('with_instructions', 'Q2',  'BIZ_1 92 (up to 103), BIZ_2 108 (up to 121), BIZ_3 99 (up to 111)', 'Partial', 'Adds upper bounds, but drops the All businesses total (285) and the note that businesses do not add up'),
('with_instructions', 'Q3',  'MRR USD 3,956.01, up to USD 4,441.65; lifetime USD 959.08', 'Correct', 'Right numbers, adds the range; does not give the 24.2% share'),
('with_instructions', 'Q4',  'Jun-Aug 2026: All 13.2%, annual 9.8%, lifetime 4.8%, monthly 27.2%, all with CIs', 'Correct', 'Right numbers and period'),
('with_instructions', 'Q5',  'BIZ_1 highest 15.7% (CI 9.6-24.7%); says difference vs company-wide may be meaningful as CIs do not fully overlap', 'Wrong', 'Misreads the interval rule: the intervals overlap; also compares BIZ_1 with a total that includes it'),
('with_instructions', 'Q6',  'New content 29.2% of content events in August 2026', 'Partial', 'Right number, but drops the comparison with catalog share (7.7%)'),
('with_instructions', 'Q7',  'Catalog reach 75.5% (CI 70.2-80.2%); says 75.5% of users saw an item; calls discovery strong', 'Partial', 'Right metric and CI, but misstates the definition (items, not users), judges it strong, no content age breakdown'),
('with_instructions', 'Q8',  'May-Jul 2026 window 14.2% (CI 10.6-18.8%), described as a 3-month period', 'Correct', 'Right window, labeled; no July churn; no overlapping comparison'),
('with_instructions', 'Q9',  'Jun-Aug 2026 13.2% vs. Jan-Mar 2023 0.0%; notable increase; CIs overlap slightly', 'Partial', 'Non-overlapping windows and CIs, but compares with a tiny 2023 base and still calls it a notable increase'),
('with_instructions', 'Q10', 'MRR only available through August 2026; no number for October', 'Correct', 'No number'),
('with_instructions', 'Q11', 'Not available; placeholder durations double the actual time', 'Correct', 'No number; right reason'),
('with_instructions', 'Q12', 'MAU 401 and active paid subscribers 285 (up to 314), each defined', 'Correct', 'Gives both definitions and the range'),
('with_instructions', 'H1',  'Jan-Mar 2026 window 6.6% (CI 4.1-10.4%)', 'Correct', 'Uses and labels the 3-month window'),
('with_instructions', 'H2',  'USD 13.88 (MRR / paid subscribers), up to USD 14.15 from the upper bounds', 'Partial', 'Computes an unofficial metric without saying so; ratio of upper bounds is not a valid upper bound'),
('with_instructions', 'H3',  'September not available; latest is 401 for August 2026', 'Correct', 'No September number'),
('with_instructions', 'H4',  '285 paying customers, up to 314, plus 401 MAU, with definitions', 'Correct', 'Right number, definition and range'),
('with_instructions', 'H5',  'Older content reach 66.3% (CI 59.5-72.5%), 132 of 199', 'Partial', 'Right mapping and numbers, but no comparison with newer content (95-100%)');

num_affected_rows,num_inserted_rows
17,17


## Step E – Results summary

In [0]:
SELECT r.run_id,
       CASE WHEN q.is_trap THEN 'Trap' ELSE 'Normal' END AS question_kind,
       COUNT(*) AS questions,
       SUM(CASE WHEN r.score = 'Correct' THEN 1 ELSE 0 END) AS correct,
       SUM(CASE WHEN r.score = 'Partial' THEN 1 ELSE 0 END) AS partial,
       SUM(CASE WHEN r.score = 'Wrong' THEN 1 ELSE 0 END) AS wrong
FROM workspace.scribd_exercise.eval_genie_results r
JOIN workspace.scribd_exercise.eval_golden_questions q
  ON r.question_id = q.question_id
GROUP BY GROUPING SETS ((r.run_id, CASE WHEN q.is_trap THEN 'Trap' ELSE 'Normal' END), (r.run_id))
ORDER BY r.run_id, question_kind;

run_id,question_kind,questions,correct,partial,wrong
baseline,null,17,10,6,1
baseline,Normal,9,5,4,0
baseline,Trap,8,5,2,1
with_instructions,null,17,10,6,1
with_instructions,Normal,9,4,4,1
with_instructions,Trap,8,6,2,0


**Result:** Same total in both runs (10 Correct, 6 Partial, 1 Wrong of 17), different failures. Instructions fixed the traps (Wrong 1 → 0; "July churn", the "churn increased" premise, ranges and session time handled correctly) but the normal questions regressed (Correct 5 → 4), because the trimmed instructions dropped context: totals in breakdowns, the catalog-share comparison, and how to read overlapping intervals. 

### Instructions used in run 3 
SCOPE
Answer only from the metric views in this space. Every number must come from a view column. Never aggregate rows into a new metric, and never divide one metric by another to create a new one (e.g., revenue per customer); say it isn't an official metric and give the official metrics it would use. If a metric doesn't exist, say so and suggest the closest official one.

TIME
- Data is reliable only through August 2026. For September 2026 or later, say so and give no number.
- If no period is given, use the latest complete period (August 2026; churn: the 3-month window ending August 31, 2026) and say which.
- Always state the period. Trends start in January 2023; for comparisons over time, use periods from 2025 onward, since earlier periods have too few customers.

BUSINESSES
- For company-wide numbers, always use the business_id = 'All businesses' row; never compute totals yourself.
- In any breakdown by business, also show the 'All businesses' row.
- MAU and active paid subscribers: businesses do NOT add up to the total (people can belong to several); say so. MRR: businesses add up. Rates: never add or average across businesses.
- Never compare a business with 'All businesses' (the total includes that business); compare businesses with each other.

CHURN
- Churn exists only as 3-month windows; there is no monthly churn. For a month, give the window ending in that month with its full label (e.g., "January–March 2026"), never "<month> churn".
- To compare over time, compare the latest window with the previous window that doesn't overlap (3 months earlier). Use billing_cycle = 'All billing cycles' unless asked.

UNCERTAINTY
- Churn and catalog reach have 95% intervals (_ci_low, _ci_high). Two intervals overlap when the lower bound of each is below the upper bound of the other. If they overlap, say the difference may be chance and is not reliable.
- For active paid subscribers and MRR, report the upper bound too ("X, up to Y").
- Describe differences as associations. Don't call results strong, healthy or weak; report the numbers and comparisons. Check assumptions in the question before accepting them.

METRICS (always show the companion)
- Catalog reach = share of live catalog ITEMS with at least one content event in the month (not a share of users). When asked about content age or the backlist, show all three age bands.
- Content engagement share by age: always show the band's catalog share next to it.

TERMS
- "Users": ask whether MAU or paid subscribers, or give both with definitions.
- "Revenue" = MRR (listed price; no collected revenue). "Discovery" = catalog reach.

NOT AVAILABLE (explain, don't calculate)
Average session time (placeholder durations double it), completion rate or funnel (completions lack opens), trial conversion (stale trials), collected revenue (no payment data), search success (no query text), monthly churn, content rankings (engagement too even), revenue per customer or any other ratio of metrics.

STYLE
Number and period first, then the comparison or companion number that gives it meaning, then caveats. Percentages with one decimal; USD with two decimals.

In [0]:
INSERT INTO workspace.scribd_exercise.eval_genie_results VALUES
('run_3', 'Q1',  'MAU 401 in August 2026', 'Correct', 'Right number and period'),
('run_3', 'Q2',  'All 285 (up to 314); BIZ_1 92 (103), BIZ_2 108 (121), BIZ_3 99 (111); businesses do not add up', 'Correct', 'Total, ranges and the do-not-add-up note'),
('run_3', 'Q3',  'MRR USD 3,956.01 up to USD 4,441.65; lifetime USD 959.08, about a quarter', 'Correct', 'Right numbers, range and share'),
('run_3', 'Q4',  'Jun-Aug 2026: All 13.2%, annual 9.8%, lifetime 4.8%, monthly 27.2%, all with CIs', 'Correct', 'Right numbers and period'),
('run_3', 'Q5',  'BIZ_1 highest 15.73% (CI 9.61-24.69%), BIZ_2 12.96%, BIZ_3 11.11%; total excluded', 'Partial', 'No longer compares with the total, but does not say the intervals overlap'),
('run_3', 'Q6',  'New content 29.2% of engagement vs. 7.7% of catalog; all three bands', 'Correct', 'Comparison with catalog share restored'),
('run_3', 'Q7',  'Catalog reach 75.5% (CI 70.2-80.2%); new 100%, older 66.3%', 'Correct', 'Names the metric, right definition, age bands, no judgment'),
('run_3', 'Q8',  'May-Jul 2026 window 14.2% (CI 10.6-18.8%); monthly churn not available', 'Correct', 'Labeled 3-month window, explains no monthly churn'),
('run_3', 'Q9',  'Jun-Aug 13.2% vs. previous non-overlapping Mar-May 10.9%; may be chance as intervals overlap', 'Correct', 'Right windows, checks the premise, interval caveat'),
('run_3', 'Q10', 'October not available; latest August 2026 USD 3,956.01 up to USD 4,441.65', 'Correct', 'No number for October'),
('run_3', 'Q11', 'Not an official metric; placeholder durations double it', 'Correct', 'No number; right reason'),
('run_3', 'Q12', 'MAU 401 and active paid subscribers 285 (up to 314)', 'Correct', 'Gives both, clearly labeled, with the range'),
('run_3', 'H1',  'Jan-Mar 2026 window 6.6% (CI 4.1-10.4%)', 'Correct', 'Labeled 3-month window'),
('run_3', 'H2',  'Not an official metric, cannot calculate; offers MRR and paid subscribers separately', 'Correct', 'Refuses the unofficial ratio and points to official metrics'),
('run_3', 'H3',  'Empty result table, no explanation', 'Partial', 'No number, but does not explain that data stops in August 2026'),
('run_3', 'H4',  'MAU 401 and active paid subscribers 285 (up to 314), August 2026', 'Correct', 'Paying customers with range, labeled'),
('run_3', 'H5',  'Older content 66.3% (CI 59.5-72.5%), 69.6% of catalog, 38.0% of engagement; newer content higher reach; seen by a majority of users', 'Partial', 'Right numbers and comparison, but misstates reach as a share of users');

num_affected_rows,num_inserted_rows
17,17


In [0]:
SELECT r.run_id,
       CASE WHEN q.is_trap THEN 'Trap' ELSE 'Normal' END AS question_kind,
       COUNT(*) AS questions,
       SUM(CASE WHEN r.score = 'Correct' THEN 1 ELSE 0 END) AS correct,
       SUM(CASE WHEN r.score = 'Partial' THEN 1 ELSE 0 END) AS partial,
       SUM(CASE WHEN r.score = 'Wrong' THEN 1 ELSE 0 END) AS wrong
FROM workspace.scribd_exercise.eval_genie_results r
JOIN workspace.scribd_exercise.eval_golden_questions q
  ON r.question_id = q.question_id
GROUP BY GROUPING SETS ((r.run_id, CASE WHEN q.is_trap THEN 'Trap' ELSE 'Normal' END), (r.run_id))
ORDER BY r.run_id, question_kind;

run_id,question_kind,questions,correct,partial,wrong
baseline,null,17,10,6,1
baseline,Normal,9,5,4,0
baseline,Trap,8,5,2,1
run_3,null,17,14,3,0
run_3,Normal,9,7,2,0
run_3,Trap,8,7,1,0
with_instructions,null,17,10,6,1
with_instructions,Normal,9,4,4,1
with_instructions,Trap,8,6,2,0
